# 10 — Feature Engineering for Next-Basket Prediction

## 1. Objective

In Notebook 09, we created a new list of products for each customer.

This list contains:

- Products the customer bought before
- New products the customer never bought before

Now we need to prepare information about each customer and each product so that a machine learning model can learn from it.

For every customer-product pair, we will create features such as:

- How often the customer buys this product
- How recently the customer bought this product
- How often the customer usually reorders products
- How popular the product is
- How often other customers reorder the product
- Whether the product belongs to an aisle the customer often buys from
- Whether the product was suggested using the aisle method
- Whether the product was suggested using the co-purchase method
- Information about the customer's recent orders

For products the customer never bought before, some information will not exist.

For example, we cannot calculate:

- How many times the customer bought the product
- When the customer last bought the product
- How often the customer reordered the product

For these new products, we will use `0` for these values and create a feature that tells the model that the product is new to the customer.

At the end of this notebook, each row will represent:

Customer + Product + Target Order

The target will be:

- `1` if the customer bought the product in the target order
- `0` if the customer did not buy the product

The final table will be used in the next notebook to train a machine learning model that predicts which products are most likely to appear in the customer's next basket.

## 2. Load the Candidate Data

We start by loading the final candidate table created in Notebook 09.

Each row represents one customer and one possible product for the customer's next order.

The table also contains the target:

- `1` if the customer actually bought the product
- `0` if the customer did not buy the product

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

candidates_df = spark.table(
    "workspace.ml_data.next_basket_candidates"
)

display(candidates_df.limit(10))

user_id,target_order_id,product_id,is_reorder_candidate,is_aisle_candidate,is_copurchase_candidate,aisle_candidate_rank,copurchase_candidate_rank,candidate_source_count,candidate_source,target_purchased
73499,1286613,15269,1,0,0,null,null,1,reorder,0
95951,2789939,7021,1,0,0,null,null,1,reorder,0
148177,3081943,33846,1,0,0,null,null,1,reorder,0
107811,2230145,15420,1,0,0,null,null,1,reorder,0
170091,504018,37158,1,0,0,null,null,1,reorder,0
154921,565130,44728,1,0,0,null,null,1,reorder,0
152631,1843393,12980,1,0,0,null,null,1,reorder,0
192125,2964943,38374,1,0,0,null,null,1,reorder,0
182595,817545,10070,1,0,0,null,null,1,reorder,0
95268,1993442,18918,1,0,0,null,null,1,reorder,0


## 3. Load the Existing Features

In Notebook 07, we already calculated many useful features about customers and products.

For example:

- How often a customer places orders
- Average basket size
- Customer reorder rate
- Product popularity
- Product reorder rate
- How often a customer bought a specific product

We load that table so we can reuse these features instead of calculating everything again.

In [0]:
previous_features_df = spark.table(
    "workspace.ml_data.reorder_features"
)

print("Previous feature rows:", previous_features_df.count())
print("Previous feature columns:", len(previous_features_df.columns))

Previous feature rows: 8474661
Previous feature columns: 74


## 4. Reuse Customer Features

The previous feature table contains many rows for the same customer because each customer has several products.

Customer information does not depend on the product.

For example:

- Number of previous orders
- Average basket size
- Customer reorder rate
- Average time between orders
- Usual shopping day and hour

We therefore create one customer feature table with one row per customer.

In [0]:
customer_feature_columns = [
    "customer_prior_orders",
    "customer_total_products",
    "customer_unique_products",
    "customer_avg_basket_size",
    "customer_reorder_rate",
    "customer_avg_days_between_orders",
    "customer_std_days_between_orders",
    "customer_avg_order_hour",
    "customer_active_days_of_week",
    "customer_last_basket_size",
    "customer_avg_last_3_basket_size",
    "customer_basket_size_trend",
    "customer_preferred_dow",
    "customer_preferred_day_share",
    "customer_preferred_hour",
    "customer_preferred_hour_share"
]

customer_features_df = (
    previous_features_df
    .select(
        "user_id",
        *customer_feature_columns
    )
    .distinct()
)

print("Customer feature rows:", customer_features_df.count())

Customer feature rows: 131209


## 5. Reuse Product Features

Some features describe the product itself and are the same for every customer.

For example:

- Number of purchases
- Number of customers who bought the product
- Product reorder rate
- Average cart position
- Aisle
- Department

We therefore create one product feature table with one row per product.

In [0]:
product_feature_columns = [
    "product_purchase_count",
    "product_unique_customers",
    "product_reorder_rate",
    "product_avg_cart_position",
    "product_name",
    "aisle_id",
    "department_id",
    "product_purchases_per_customer",
    "product_order_share",
    "product_preferred_dow",
    "product_preferred_day_share",
    "product_preferred_hour",
    "product_preferred_hour_share"
]

product_features_df = (
    previous_features_df
    .select(
        "product_id",
        *product_feature_columns
    )
    .distinct()
)

print("Product feature rows:", product_features_df.count())

Product feature rows: 49468


## 6. Reuse Customer-Product History

Some features describe the history between one customer and one product.

For example:

- How many times the customer bought the product
- How often the customer reordered it
- When the customer first and last bought it
- Whether the product was bought in the last order
- How many times it appeared in the last 3 or 5 orders
- How regularly the customer buys the product

These features only exist for products the customer bought before.

For new products, there is no previous customer-product history.

In [0]:
customer_product_feature_columns = [
    "user_product_order_count",
    "user_product_reorder_rate",
    "user_product_avg_cart_position",
    "user_product_first_order",
    "user_product_last_order",
    "user_product_order_share",
    "orders_since_last_product_purchase",
    "user_product_avg_order_gap",
    "user_product_due_score",
    "bought_in_last_order",
    "purchases_last_3_orders",
    "purchases_last_5_orders",
    "purchase_rate_last_3_orders",
    "purchase_rate_last_5_orders",
    "recent_purchase_momentum",
    "user_product_max_streak",
    "user_product_current_streak"
]

customer_product_features_df = (
    previous_features_df
    .select(
        "user_id",
        "product_id",
        *customer_product_feature_columns
    )
    .distinct()
)

print(
    "Customer-product feature rows:",
    customer_product_features_df.count()
)

Customer-product feature rows: 8474661


## 7. Add Features to the Candidate Table

We now add the existing customer, product, and customer-product features to every candidate.

For products the customer bought before, customer-product history will be available.

For new products, customer-product history will be empty because the customer has never purchased the product before.

We will handle these empty values in the next step.

In [0]:
candidate_features_df = (
    candidates_df

    # Add customer information
    .join(
        customer_features_df,
        on="user_id",
        how="left"
    )

    # Add product information
    .join(
        product_features_df,
        on="product_id",
        how="left"
    )

    # Add customer-product history
    .join(
        customer_product_features_df,
        on=["user_id", "product_id"],
        how="left"
    )
)

print("Candidate feature rows:", candidate_features_df.count())
print("Candidate feature columns:", len(candidate_features_df.columns))

Candidate feature rows: 13519765
Candidate feature columns: 57


## 8. Check Missing Customer-Product History

Some candidates are new products that the customer has never purchased before.

For these products, customer-product history does not exist.

We first check how many empty values appear in these historical features before replacing them.

In [0]:
missing_history_df = candidate_features_df.agg(
    *[
        F.sum(
            F.when(F.col(column).isNull(), 1).otherwise(0)
        ).alias(column)
        for column in customer_product_feature_columns
    ]
)

display(missing_history_df)

user_product_order_count,user_product_reorder_rate,user_product_avg_cart_position,user_product_first_order,user_product_last_order,user_product_order_share,orders_since_last_product_purchase,user_product_avg_order_gap,user_product_due_score,bought_in_last_order,purchases_last_3_orders,purchases_last_5_orders,purchase_rate_last_3_orders,purchase_rate_last_5_orders,recent_purchase_momentum,user_product_max_streak,user_product_current_streak
5045104,5045104,5045104,5045104,5045104,5045104,5045104,5045104,5045104,5045104,5045104,5045104,5045104,5045104,5045104,5045104,5045104


## 9. Handle New Products

New products do not have customer-product purchase history.

This is expected because the customer has never bought these products before.

For these candidates, we replace the missing historical values with `0`.

We also create `is_new_to_customer`:

- `1` if the customer has never purchased the product
- `0` if the customer purchased the product before

This helps the machine learning model distinguish new products from reorder products.

In [0]:
# Create an indicator for new products
candidate_features_clean_df = (
    candidate_features_df
    .withColumn(
        "is_new_to_customer",
        F.when(
            F.col("is_reorder_candidate") == 0,
            1
        ).otherwise(0)
    )
)

# Replace missing customer-product history with 0
candidate_features_clean_df = (
    candidate_features_clean_df
    .fillna(
        0,
        subset=customer_product_feature_columns
    )
)

# Check the result
display(
    candidate_features_clean_df
    .groupBy("is_new_to_customer")
    .count()
    .orderBy("is_new_to_customer")
)

is_new_to_customer,count
0,8474661
1,5045104


## 10. Add Target Order Information

Some features describe the customer's target order.

For example:

- Day of the week
- Hour of the day
- Days since the previous order
- Difference between the current shopping time and the customer's usual shopping rhythm
- Expected basket size

This information is the same for all candidate products of the same customer.

We reuse these features from Notebook 07.

In [0]:
target_feature_columns = [
    "target_order_dow",
    "target_order_hour",
    "target_days_since_prior_order",
    "order_timing_deviation_days",
    "order_timing_ratio",
    "expected_basket_size",
    "recent_vs_usual_basket_ratio",
    "target_order_number",
    "has_order_timing_ratio"
]

target_features_df = (
    previous_features_df
    .select(
        "user_id",
        "target_order_id",
        *target_feature_columns
    )
    .distinct()
)

print("Target feature rows:", target_features_df.count())

Target feature rows: 131209


## 11. Add Target Order Features

We now add the target order information to every candidate product.

All candidate products belonging to the same customer and target order receive the same target order features.

These features help the model understand the situation in which the next purchase happens.

In [0]:
candidate_features_clean_df = (
    candidate_features_clean_df
    .join(
        target_features_df,
        on=["user_id", "target_order_id"],
        how="left"
    )
)

print(
    "Rows after adding target features:",
    candidate_features_clean_df.count()
)

print(
    "Columns after adding target features:",
    len(candidate_features_clean_df.columns)
)

Rows after adding target features: 13519765
Columns after adding target features: 67


## 12. Add Aisle and Department Preferences

We now measure how much each customer usually buys from the aisle and department of each candidate product.

For example, if a customer often buys products from the same aisle as a candidate product, that candidate may be more relevant.

We add:

- `customer_aisle_affinity` — how much the customer buys from this aisle
- `customer_department_affinity` — how much the customer buys from this department

If the customer has never bought anything from that aisle or department, the value is set to `0`.

In [0]:
# Customer preference for each aisle
customer_aisle_affinity_df = (
    previous_features_df
    .select(
        "user_id",
        "aisle_id",
        "customer_aisle_affinity"
    )
    .distinct()
)

# Customer preference for each department
customer_department_affinity_df = (
    previous_features_df
    .select(
        "user_id",
        "department_id",
        "customer_department_affinity"
    )
    .distinct()
)

# Add aisle and department preferences to every candidate
candidate_features_clean_df = (
    candidate_features_clean_df

    .join(
        customer_aisle_affinity_df,
        on=["user_id", "aisle_id"],
        how="left"
    )

    .join(
        customer_department_affinity_df,
        on=["user_id", "department_id"],
        how="left"
    )

    .fillna({
        "customer_aisle_affinity": 0.0,
        "customer_department_affinity": 0.0
    })
)

print(
    "Rows after adding category preferences:",
    candidate_features_clean_df.count()
)

print(
    "Columns after adding category preferences:",
    len(candidate_features_clean_df.columns)
)

Rows after adding category preferences: 13519765
Columns after adding category preferences: 69


## 13. Add Shopping-Time Features

We compare the target order with the customer's and product's usual shopping time.

We create features that tell us:

- Whether the target day matches the customer's usual shopping day
- Whether the target day matches the product's usual purchase day
- How far the target day is from the usual day
- How far the target hour is from the usual shopping hour

A smaller distance means the target order is closer to the usual shopping pattern.

In [0]:
# Difference between target day and usual day
customer_day_diff = F.abs(
    F.col("target_order_dow") - F.col("customer_preferred_dow")
)

product_day_diff = F.abs(
    F.col("target_order_dow") - F.col("product_preferred_dow")
)

# Difference between target hour and usual hour
customer_hour_diff = F.abs(
    F.col("target_order_hour") - F.col("customer_preferred_hour")
)

product_hour_diff = F.abs(
    F.col("target_order_hour") - F.col("product_preferred_hour")
)

candidate_features_clean_df = (
    candidate_features_clean_df

    # Exact day matches
    .withColumn(
        "customer_day_match",
        F.when(
            F.col("target_order_dow") == F.col("customer_preferred_dow"),
            1
        ).otherwise(0)
    )

    .withColumn(
        "product_day_match",
        F.when(
            F.col("target_order_dow") == F.col("product_preferred_dow"),
            1
        ).otherwise(0)
    )

    # Circular day distance
    .withColumn(
        "customer_day_distance",
        F.least(
            customer_day_diff,
            F.lit(7) - customer_day_diff
        )
    )

    .withColumn(
        "product_day_distance",
        F.least(
            product_day_diff,
            F.lit(7) - product_day_diff
        )
    )

    # Circular hour distance
    .withColumn(
        "customer_hour_distance_circular",
        F.least(
            customer_hour_diff,
            F.lit(24) - customer_hour_diff
        )
    )

    .withColumn(
        "product_hour_distance_circular",
        F.least(
            product_hour_diff,
            F.lit(24) - product_hour_diff
        )
    )
)

print(
    "Rows after adding shopping-time features:",
    candidate_features_clean_df.count()
)

print(
    "Columns after adding shopping-time features:",
    len(candidate_features_clean_df.columns)
)

Rows after adding shopping-time features: 13519765
Columns after adding shopping-time features: 75


## 14. Add Recent Basket Features

We now add information about the customer's most recent basket.

These features tell us whether a candidate product is related to what the customer bought recently.

We add:

- Whether the product was bought in the last order
- Whether its aisle appeared in the last order
- Whether its department appeared in the last order
- How similar the customer's recent baskets are
- A score that combines recent basket information with basket similarity

For new products, the product itself was not bought before, but its aisle or department may still have appeared in the customer's last basket.

In [0]:
basket_feature_columns = [
    "aisle_in_last_basket",
    "department_in_last_basket",
    "customer_basket_stability",
    "repeat_stability_signal",
    "last_basket_context_level",
    "stability_weighted_context"
]

basket_features_df = (
    previous_features_df
    .select(
        "user_id",
        "product_id",
        *basket_feature_columns
    )
    .distinct()
)

print(
    "Basket feature rows:",
    basket_features_df.count()
)

Basket feature rows: 8474661


### 14.1 — Prepare Last-Basket Information

The old basket features were created only for products the customer bought before.

For new products, we still need to know:

- Whether the product's aisle appeared in the customer's last basket
- Whether the product's department appeared in the customer's last basket
- How similar the customer's recent baskets usually are

We therefore create separate customer, aisle, and department tables that can also be used for new products.

In [0]:
# Customer basket stability
customer_basket_stability_df = (
    previous_features_df
    .select(
        "user_id",
        "customer_basket_stability"
    )
    .distinct()
)

# Whether an aisle appeared in the customer's last basket
customer_last_basket_aisles_df = (
    previous_features_df
    .select(
        "user_id",
        "aisle_id",
        "aisle_in_last_basket"
    )
    .groupBy(
        "user_id",
        "aisle_id"
    )
    .agg(
        F.max("aisle_in_last_basket")
        .alias("aisle_in_last_basket")
    )
)

# Whether a department appeared in the customer's last basket
customer_last_basket_departments_df = (
    previous_features_df
    .select(
        "user_id",
        "department_id",
        "department_in_last_basket"
    )
    .groupBy(
        "user_id",
        "department_id"
    )
    .agg(
        F.max("department_in_last_basket")
        .alias("department_in_last_basket")
    )
)

print(
    "Customer basket stability rows:",
    customer_basket_stability_df.count()
)

print(
    "Customer-aisle rows:",
    customer_last_basket_aisles_df.count()
)

print(
    "Customer-department rows:",
    customer_last_basket_departments_df.count()
)

Customer basket stability rows: 131209
Customer-aisle rows: 3647699
Customer-department rows: 1421165


### 14.2 — Add Recent Basket Information

We now add recent basket information to every candidate product.

For each candidate, we check:

- Whether the product itself was bought in the last order
- Whether its aisle appeared in the last order
- Whether its department appeared in the last order
- How similar the customer's recent baskets are

We also create a simple context level:

- `3` if the exact product was bought in the last order
- `2` if the product was not bought, but its aisle appeared
- `1` if only its department appeared
- `0` if none of these happened

This also works for new products because their aisle or department may still match the customer's recent basket.

In [0]:
candidate_features_clean_df = (
    candidate_features_clean_df

    # Add customer basket stability
    .join(
        customer_basket_stability_df,
        on="user_id",
        how="left"
    )

    # Add last-basket aisle information
    .join(
        customer_last_basket_aisles_df,
        on=["user_id", "aisle_id"],
        how="left"
    )

    # Add last-basket department information
    .join(
        customer_last_basket_departments_df,
        on=["user_id", "department_id"],
        how="left"
    )

    # Missing means the aisle or department was not in the last basket
    .fillna({
        "customer_basket_stability": 0.0,
        "aisle_in_last_basket": 0,
        "department_in_last_basket": 0
    })

    # Combine product and basket information
    .withColumn(
        "repeat_stability_signal",
        F.col("customer_basket_stability")
        * F.col("bought_in_last_order")
    )

    .withColumn(
        "last_basket_context_level",
        F.when(F.col("bought_in_last_order") == 1, 3)
        .when(F.col("aisle_in_last_basket") == 1, 2)
        .when(F.col("department_in_last_basket") == 1, 1)
        .otherwise(0)
    )

    .withColumn(
        "stability_weighted_context",
        (
            F.col("last_basket_context_level") / 3
        ) * F.col("customer_basket_stability")
    )
)

print(
    "Rows after adding recent basket features:",
    candidate_features_clean_df.count()
)

print(
    "Columns after adding recent basket features:",
    len(candidate_features_clean_df.columns)
)

Rows after adding recent basket features: 13519765
Columns after adding recent basket features: 81


## 15. Prepare Candidate-Source Features

We already know why each product was added to the candidate list.

The candidate table contains:

- `is_reorder_candidate`
- `is_aisle_candidate`
- `is_copurchase_candidate`
- `candidate_source_count`

It also contains ranks for aisle and co-purchase candidates.

A smaller rank means a stronger suggestion.

We create two simple scores:

- `aisle_candidate_score`
- `copurchase_candidate_score`

A product ranked first receives the highest score.

If the product did not come from that method, its score is `0`.

In [0]:
candidate_features_clean_df = (
    candidate_features_clean_df

    .withColumn(
        "aisle_candidate_score",
        F.when(
            F.col("is_aisle_candidate") == 1,
            1.0 / F.col("aisle_candidate_rank")
        ).otherwise(0.0)
    )

    .withColumn(
        "copurchase_candidate_score",
        F.when(
            F.col("is_copurchase_candidate") == 1,
            1.0 / F.col("copurchase_candidate_rank")
        ).otherwise(0.0)
    )

    # Replace unused ranks with 0
    .fillna({
        "aisle_candidate_rank": 0,
        "copurchase_candidate_rank": 0
    })
)

display(
    candidate_features_clean_df
    .select(
        "user_id",
        "product_id",
        "is_reorder_candidate",
        "is_aisle_candidate",
        "is_copurchase_candidate",
        "aisle_candidate_rank",
        "aisle_candidate_score",
        "copurchase_candidate_rank",
        "copurchase_candidate_score"
    )
    .limit(20)
)

user_id,product_id,is_reorder_candidate,is_aisle_candidate,is_copurchase_candidate,aisle_candidate_rank,aisle_candidate_score,copurchase_candidate_rank,copurchase_candidate_score
170091,37158,1,0,0,0,0.0,0,0.0
33477,28523,1,0,0,0,0.0,0,0.0
29367,1003,1,0,0,0,0.0,0,0.0
30497,31273,1,0,0,0,0.0,0,0.0
151632,30450,1,0,0,0,0.0,0,0.0
128642,7978,1,0,0,0,0.0,0,0.0
72326,32133,1,0,0,0,0.0,0,0.0
199841,30720,1,0,0,0,0.0,0,0.0
148177,33846,1,0,0,0,0.0,0,0.0
95268,18918,1,0,0,0,0.0,0,0.0


### 15.1 — Check Candidate Scores

We check a few new-product candidates to make sure the aisle and co-purchase scores were created correctly.

For these products, the score should be greater than `0` when the product came from that candidate-generation method.

In [0]:
display(
    candidate_features_clean_df
    .filter(
        F.col("is_new_to_customer") == 1
    )
    .select(
        "user_id",
        "product_id",
        "is_aisle_candidate",
        "aisle_candidate_rank",
        F.round("aisle_candidate_score", 4)
         .alias("aisle_candidate_score"),
        "is_copurchase_candidate",
        "copurchase_candidate_rank",
        F.round("copurchase_candidate_score", 4)
         .alias("copurchase_candidate_score")
    )
    .limit(20)
)

user_id,product_id,is_aisle_candidate,aisle_candidate_rank,aisle_candidate_score,is_copurchase_candidate,copurchase_candidate_rank,copurchase_candidate_score
1031,42139,1,17,0.0588,0,0,0.0
799,19348,1,14,0.0714,0,0,0.0
87,13176,1,30,0.0333,1,1,1.0
52,44375,1,24,0.0417,0,0,0.0
157,28204,1,18,0.0556,0,0,0.0
599,23909,1,13,0.0769,0,0,0.0
956,44375,1,16,0.0625,0,0,0.0
96,5876,1,22,0.0455,0,0,0.0
558,21616,1,19,0.0526,0,0,0.0
591,16290,1,9,0.1111,0,0,0.0


## 16. Add Reorder-Cycle Information

For some previously purchased products, we can estimate how regularly the customer buys the product.

For example, a customer may usually buy the same product every 3 or 4 orders.

We add `has_reorder_cycle`:

- `1` if we have enough previous purchases to calculate a buying cycle
- `0` if we do not

For new products, the value is also `0` because the customer has never purchased them before.

In [0]:
reorder_cycle_df = (
    previous_features_df
    .select(
        "user_id",
        "product_id",
        "has_reorder_cycle"
    )
    .distinct()
)

candidate_features_clean_df = (
    candidate_features_clean_df
    .join(
        reorder_cycle_df,
        on=["user_id", "product_id"],
        how="left"
    )
    .fillna({
        "has_reorder_cycle": 0
    })
)

display(
    candidate_features_clean_df
    .groupBy(
        "is_new_to_customer",
        "has_reorder_cycle"
    )
    .count()
    .orderBy(
        "is_new_to_customer",
        "has_reorder_cycle"
    )
)

is_new_to_customer,has_reorder_cycle,count
0,0,5083521
0,1,3391140
1,0,5045104


## 17. Check for Missing Values

Before using the data for machine learning, we check whether any columns still contain empty values.

Some empty values were expected earlier for new products, but we have now handled them.

This check helps us find any remaining problems before saving the final feature table.

In [0]:
missing_values_df = (
    candidate_features_clean_df
    .agg(
        *[
            F.sum(
                F.when(F.col(column).isNull(), 1).otherwise(0)
            ).alias(column)
            for column in candidate_features_clean_df.columns
        ]
    )
)

display(missing_values_df)

user_id,product_id,department_id,aisle_id,target_order_id,is_reorder_candidate,is_aisle_candidate,is_copurchase_candidate,aisle_candidate_rank,copurchase_candidate_rank,candidate_source_count,candidate_source,target_purchased,customer_prior_orders,customer_total_products,customer_unique_products,customer_avg_basket_size,customer_reorder_rate,customer_avg_days_between_orders,customer_std_days_between_orders,customer_avg_order_hour,customer_active_days_of_week,customer_last_basket_size,customer_avg_last_3_basket_size,customer_basket_size_trend,customer_preferred_dow,customer_preferred_day_share,customer_preferred_hour,customer_preferred_hour_share,product_purchase_count,product_unique_customers,product_reorder_rate,product_avg_cart_position,product_name,product_purchases_per_customer,product_order_share,product_preferred_dow,product_preferred_day_share,product_preferred_hour,product_preferred_hour_share,user_product_order_count,user_product_reorder_rate,user_product_avg_cart_position,user_product_first_order,user_product_last_order,user_product_order_share,orders_since_last_product_purchase,user_product_avg_order_gap,user_product_due_score,bought_in_last_order,purchases_last_3_orders,purchases_last_5_orders,purchase_rate_last_3_orders,purchase_rate_last_5_orders,recent_purchase_momentum,user_product_max_streak,user_product_current_streak,is_new_to_customer,target_order_dow,target_order_hour,target_days_since_prior_order,order_timing_deviation_days,order_timing_ratio,expected_basket_size,recent_vs_usual_basket_ratio,target_order_number,has_order_timing_ratio,customer_aisle_affinity,customer_department_affinity,customer_day_match,product_day_match,customer_day_distance,product_day_distance,customer_hour_distance_circular,product_hour_distance_circular,customer_basket_stability,aisle_in_last_basket,department_in_last_basket,repeat_stability_signal,last_basket_context_level,stability_weighted_context,aisle_candidate_score,copurchase_candidate_score,has_reorder_cycle
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 18. Check the Final Feature Dataset

Before saving the table, we check that:

- The number of rows is correct
- Each customer-product-target order appears only once
- There are no duplicate rows
- The target still contains the correct number of positive cases
- The number of columns is correct

This is the final check before using the table for machine learning.

In [0]:
final_check_df = (
    candidate_features_clean_df
    .agg(
        F.count("*").alias("total_rows"),

        F.countDistinct(
            "user_id",
            "target_order_id",
            "product_id"
        ).alias("unique_rows"),

        F.sum("target_purchased").alias("positive_targets")
    )
    .withColumn(
        "duplicate_rows",
        F.col("total_rows") - F.col("unique_rows")
    )
    .withColumn(
        "positive_percentage",
        F.round(
            F.col("positive_targets")
            / F.col("total_rows") * 100,
            2
        )
    )
    .withColumn(
        "total_columns",
        F.lit(len(candidate_features_clean_df.columns))
    )
)

display(final_check_df)

total_rows,unique_rows,positive_targets,duplicate_rows,positive_percentage,total_columns
13519765,13519765,873920,0,6.46,84


## 19. Save the Final Feature Table

The final dataset is now ready for machine learning.

Each row represents one customer, one candidate product, and one target order.

The table contains 84 columns describing the customer, the product, their previous purchase history, the candidate source, shopping time, and recent basket behavior.

We save this table so the next notebook can load it directly for model training.

In [0]:
(
    candidate_features_clean_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.ml_data.next_basket_features"
    )
)

print("Next-basket feature table saved successfully:")
print("workspace.ml_data.next_basket_features")

Next-basket feature table saved successfully:
workspace.ml_data.next_basket_features


## 20. Verify the Saved Feature Table

We reload the saved table and check that it contains the expected number of rows, customers, positive targets, and columns.

This confirms that the feature table was saved correctly.

In [0]:
saved_features_df = spark.table(
    "workspace.ml_data.next_basket_features"
)

saved_summary_df = (
    saved_features_df
    .agg(
        F.count("*").alias("saved_rows"),
        F.countDistinct("user_id").alias("saved_customers"),
        F.sum("target_purchased").alias("positive_targets")
    )
    .withColumn(
        "total_columns",
        F.lit(len(saved_features_df.columns))
    )
)

display(saved_summary_df)

saved_rows,saved_customers,positive_targets,total_columns
13519765,131209,873920,84


## 21. Conclusion

In this notebook, we prepared the final data that will be used to train the next-basket prediction model.

We started with the candidate products created in Notebook 09.

The candidates include:

- Products the customer bought before
- New products suggested from favorite aisles
- New products suggested from co-purchase relationships

We then added information about:

- The customer's shopping behavior
- The product's general behavior
- The customer's previous history with the product
- Whether the product is new to the customer
- Why the product was added as a candidate
- The customer's aisle and department preferences
- The target order time
- Recent basket behavior

For new products, previous customer-product information does not exist, so these values were set to `0`.

The final dataset contains:

- 13,519,765 rows
- 131,209 customers
- 84 columns
- 873,920 products that were actually purchased in the target basket
- 6.46% positive cases
- 0 duplicate rows
- 0 missing values

The final table was saved as:

`workspace.ml_data.next_basket_features`

This table is ready to be used in the next notebook to train and compare machine learning models for next-basket prediction.